[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C56_Detection_Augmentation_Course/04_pipeline/04_aug_pipeline.ipynb)

# 04 · 增强流水线工程（算子顺序 / 触发概率 / worker RNG / 吞吐瓶颈 / WBF）

目标：把一条检测增强流水线**当成一个可测量的系统**来对待——
测它的实际触发率、测它的随机性多样性、测它的吞吐上限、测它的融合质量。

路线：流水线骨架与打点 → **概率相乘的实际触发率** → 顺序为何是几何在前 →
**DataLoader worker RNG 陷阱（正确/错误两种 seed 策略）** → 吞吐瓶颈模型与预取 →
**WBF vs NMS** 与 TTA 延迟账 → ✏️ 4 道练习 → 📖 答案 → 🧪 工程胶囊。

> 心智模型：**你写的配置 ≠ 模型见到的分布**。顺序、概率、RNG、库实现，
> 四种偏差全都静默。工程上唯一可靠的姿态是：**不推理，去测量。**

本 notebook 你会亲手实现：
1. 带触发打点的增强流水线（算子 + 概率 + 顺序）
2. 联合触发分布的**精确计算**与概率反解
3. 重采样对高频扰动的抹平量化（为什么光度必须在几何之后）
4. **三种 worker seed 策略**及其多样性后果（这是一起真实事故）
5. CPU/GPU 吞吐模型、最少 worker 数、预取只削峰不提均值的证明
6. **WBF（加权框融合）**、与 NMS 的定位精度对比、TTA 的车端延迟账

## 1 · 流水线骨架：算子 + 概率 + 顺序 + 打点

先把流水线建成一个**可观测**的对象：每个算子记录触发次数，每个样本记录触发组合。
没有打点的流水线是不可调试的。

In [ ]:
import numpy as np, math, itertools, collections, json

# ---------- 样本：一张小灰度图 + 若干框（x1,y1,x2,y2）----------
def make_sample(rng, H=48, W=64, n_box=3):
    img = rng.normal(0.5, 0.15, size=(H, W))
    boxes = []
    for _ in range(n_box):
        w = rng.integers(6, 16); h = rng.integers(6, 16)
        x1 = rng.integers(0, W - w); y1 = rng.integers(0, H - h)
        boxes.append([x1, y1, x1 + w, y1 + h])
    return {'img': img, 'boxes': np.array(boxes, dtype=float)}

# ---------- 算子 ----------
class Op:
    '''一个增强算子：名字 / 触发概率 / 类别(geometric|photometric) / 变换函数。'''
    def __init__(self, name, p, kind, fn):
        self.name, self.p, self.kind, self.fn = name, p, kind, fn
    def __repr__(self):
        return '%s(p=%.2f, %s)' % (self.name, self.p, self.kind)

def op_hflip(s, rng):                      # 几何：改像素也改标注
    img, b = s['img'][:, ::-1].copy(), s['boxes'].copy()
    W = s['img'].shape[1]
    x1, x2 = b[:, 0].copy(), b[:, 2].copy()
    b[:, 0], b[:, 2] = W - x2, W - x1      # ← 必须交换 x1/x2，否则得到非法框
    return {'img': img, 'boxes': b}

def op_shift(s, rng):                      # 几何：整数平移（roll）
    dx = int(rng.integers(-6, 7))
    b = s['boxes'].copy(); b[:, [0, 2]] += dx
    return {'img': np.roll(s['img'], dx, axis=1), 'boxes': b}

def op_bright(s, rng):                     # 光度：不改标注
    return {'img': s['img'] * float(rng.uniform(0.7, 1.3)), 'boxes': s['boxes']}

def op_noise(s, rng):
    return {'img': s['img'] + rng.normal(0, 0.08, size=s['img'].shape), 'boxes': s['boxes']}

def op_blur(s, rng):                       # 光度：3 点平滑
    x = s['img']
    return {'img': (np.roll(x, 1, 1) + 2 * x + np.roll(x, -1, 1)) / 4.0, 'boxes': s['boxes']}

# ---------- 流水线 ----------
class Pipeline:
    def __init__(self, ops):
        self.ops = ops
        self.hits = collections.Counter()      # 每个算子触发次数
        self.combo = collections.Counter()     # 每个样本触发了几个算子
        self.n = 0
    def __call__(self, sample, rng):
        out, fired = sample, []
        for op in self.ops:
            if rng.random() < op.p:            # ← 每个算子独立掷骰子
                out = op.fn(out, rng); fired.append(op.name)
        self.n += 1; self.combo[len(fired)] += 1
        for f in fired: self.hits[f] += 1
        return out, fired
    def rates(self):
        return {op.name: self.hits[op.name] / max(self.n, 1) for op in self.ops}

OPS = [Op('hflip', 0.5, 'geometric', op_hflip),
       Op('shift', 0.5, 'geometric', op_shift),
       Op('bright', 0.5, 'photometric', op_bright),
       Op('noise', 0.5, 'photometric', op_noise),
       Op('blur', 0.5, 'photometric', op_blur)]
print('流水线:', OPS)

# 顺序检查：几何必须全部排在光度之前
kinds = [o.kind for o in OPS]
assert kinds.index('photometric') > max(i for i, k in enumerate(kinds) if k == 'geometric'), \
    '几何算子必须全部排在光度算子之前'
print('✅ 顺序合法：几何段', kinds.count('geometric'), '个 -> 光度段', kinds.count('photometric'), '个')

In [ ]:
rng = np.random.default_rng(0)
pipe = Pipeline(OPS)
N = 20000
for _ in range(N):
    pipe(make_sample(rng, n_box=2), rng)

print('%-10s %10s %10s' % ('算子', '配置 p', '实测触发率'))
for op in OPS:
    r = pipe.rates()[op.name]
    print('%-10s %10.2f %10.4f' % (op.name, op.p, r))
    assert abs(r - op.p) < 0.02, op.name          # 单算子：实测≈配置，符合直觉

print()
print('触发个数分布（这才是"实际强度"）:')
for k in range(len(OPS) + 1):
    c = pipe.combo[k]
    print('  触发 %d 个: %6d 次  %6.2f%%  %s' % (k, c, 100 * c / N, '█' * int(60 * c / N)))
p_all = pipe.combo[len(OPS)] / N
p_none = pipe.combo[0] / N
print()
print('全部 5 个都触发的样本占比: %.2f%%  （理论 %.2f%%）' % (100 * p_all, 100 * 0.5 ** 5))
print('一个都不触发的样本占比:   %.2f%%' % (100 * p_none))
assert abs(p_all - 0.03125) < 0.006 and abs(p_none - 0.03125) < 0.006
print()
print('⚠️  单看每个算子都是"一半样本被增强"，合起来却只有 3.1% 的样本走完全套。')
print('   你在 review 配置时脑子里想的那个"最强组合"，模型几乎没见过。')

## 2 · 概率相乘：精确计算联合触发分布，以及反解每个 p

不要用模拟去估这个分布——它可以**精确算**（k 个算子只有 2^k 种触发组合）。

In [ ]:
def trigger_profile(ps):
    '''精确计算"恰好触发 j 个算子"的概率分布，返回长度 k+1 的数组。'''
    k = len(ps)
    dist = np.zeros(k + 1)
    for mask in itertools.product([0, 1], repeat=k):
        pr = 1.0
        for bit, p in zip(mask, ps):
            pr *= p if bit else (1.0 - p)
        dist[sum(mask)] += pr
    return dist

d = trigger_profile([0.5] * 5)
assert abs(d.sum() - 1.0) < 1e-12
assert abs(d[5] - 1 / 32) < 1e-12 and abs(d[0] - 1 / 32) < 1e-12
assert abs(float((np.arange(6) * d).sum()) - 2.5) < 1e-12
print('5 个算子各 p=0.5 的精确分布:', np.round(d, 5).tolist())
print()

print('%-22s %12s %12s %12s' % ('配置', 'P(全触发)', 'P(全不触发)', 'E[触发个数]'))
for k in (3, 5, 8):
    for p in (0.3, 0.5, 0.8):
        dd = trigger_profile([p] * k)
        print('%-22s %11.2f%% %11.2f%% %12.2f'
              % ('k=%d, p=%.1f' % (k, p), 100 * dd[k], 100 * dd[0], k * p))
print()
print('⚠️  8 个算子各 p=0.5 -> 全触发概率 0.39%，四百分之一。')
print('   这就是"配置强度"与"实际强度"之间的鸿沟。')

In [ ]:
def calibrate_uniform_p(k, target_all):
    '''反解：k 个算子、希望"全部触发"的样本占 target_all，每个算子的 p 该是多少。'''
    return target_all ** (1.0 / k)

print('%-8s %-14s %-10s' % ('算子数 k', '目标联合触发率', '每个算子的 p'))
for k in (3, 5, 8):
    for t in (0.10, 0.30, 0.50):
        p = calibrate_uniform_p(k, t)
        print('%-8d %-14s %-10.4f' % (k, '%.0f%%' % (100 * t), p))
        assert abs(p ** k - t) < 1e-12

p5 = calibrate_uniform_p(5, 0.30)
assert abs(p5 - 0.30 ** 0.2) < 1e-12
print()
print('结论：想让 30%% 的样本走完 5 个算子，每个 p 要 %.3f —— 而不是直觉的 0.5/0.6。' % p5)
print()

# 嵌套 OneOf 的概率折叠：叶子算子的实际触发率
def oneof_leaf_rate(outer_p, weights):
    '''OneOf(ops, p=outer_p) 语义：以 outer_p 进入组，进入后按归一化权重选一个。'''
    w = np.asarray(weights, float)
    return outer_p * w / w.sum()

leaf = oneof_leaf_rate(0.5, [1, 1, 1])
print('OneOf([A,B,C], p=0.5) 里每个叶子的实际触发率:', np.round(leaf, 4).tolist())
assert abs(leaf[0] - 1 / 6) < 1e-12
nested = oneof_leaf_rate(0.5, [1, 1, 1])[0] * 0.6          # 外面再套一层 Compose(p=0.6)
print('再套一层 Compose(p=0.6) 之后:', round(nested, 4))
assert abs(nested - 0.1) < 1e-12
print('⚠️  配置里写着 p=0.5 的算子，实际触发率 10%%。三层嵌套后常见 2%%-3%%。')
print('✅ 唯一可靠的做法：给流水线打点，统计实际触发率，而不是读配置推理。')

## 3 · 顺序：为什么光度必须放在几何之后

**任何亚像素几何变换都是一次低通滤波。** 先注入的高频扰动会被后续重采样抹掉——
可以精确算出抹掉多少。

In [ ]:
def shift_bilinear(x, dx):
    '''沿最后一维做亚像素平移（双线性）。等价于卷积核 [1-f, f]。'''
    i = int(math.floor(dx)); f = dx - i
    return (1 - f) * np.roll(x, -i, axis=-1) + f * np.roll(x, -(i + 1), axis=-1)

r = np.random.default_rng(1)
noise = r.normal(0, 1.0, size=200000)            # 模拟"注入的高频扰动"
v0 = noise.var()

v_once = shift_bilinear(noise, 0.5).var() / v0                       # 一次半像素平移
seq = noise
for _ in range(3):                                                    # 串联三次（逐算子各 warp 一次）
    seq = shift_bilinear(seq, 0.5)
v_seq3 = seq.var() / v0
v_comp = shift_bilinear(noise, 1.5).var() / v0                        # 合成一次（等效位移 1.5px）

print('%-34s %10s %10s' % ('几何段的做法', '方差保留', '理论值'))
print('%-34s %9.3f %10.4f' % ('一次 0.5px 平移', v_once, 0.5))
print('%-34s %9.3f %10.4f' % ('串联 3 次 0.5px（逐算子 warp）', v_seq3, 20 / 64))
print('%-34s %9.3f %10.4f' % ('合成矩阵一次 warp（等效 1.5px）', v_comp, 0.5))
assert abs(v_once - 0.5) < 0.02
assert abs(v_seq3 - 0.3125) < 0.02, '三重卷积核 [1,3,3,1]/8 -> ||h||^2 = 20/64'
assert abs(v_comp - 0.5) < 0.02
print()
print('结论 A（光度要放后面）：你配置 sigma=0.10 的噪声，若放在几何之前，')
print('   串联几何后实际只剩 sigma=%.4f —— **强度打了对折**。' % (0.10 * math.sqrt(v_seq3)))
print('结论 B（几何要合成矩阵）：串联 3 次比合成 1 次多损失 %.1f%% 的高频能量，'
      % (100 * (v_comp - v_seq3) / v_comp))
print('   而高频正是小目标（8-16px 的交通标志）赖以被检出的信息。')

In [ ]:
# 越界处理必须紧跟几何段：演示"坐标系变了却没清洗"的后果
def clip_and_filter(boxes, W, H, min_area=16.0, min_visible=0.3):
    '''裁剪到画布内 + 按面积/可见比例丢弃残框。返回 (保留的框, 丢弃数)。'''
    b = boxes.copy()
    area0 = np.maximum(b[:, 2] - b[:, 0], 0) * np.maximum(b[:, 3] - b[:, 1], 0)
    b[:, [0, 2]] = np.clip(b[:, [0, 2]], 0, W)
    b[:, [1, 3]] = np.clip(b[:, [1, 3]], 0, H)
    area1 = np.maximum(b[:, 2] - b[:, 0], 0) * np.maximum(b[:, 3] - b[:, 1], 0)
    vis = np.divide(area1, np.maximum(area0, 1e-9))
    keep = (area1 >= min_area) & (vis >= min_visible)
    return b[keep], int((~keep).sum())

rr = np.random.default_rng(7)
s = make_sample(rr, n_box=6)
H, W = s['img'].shape
moved = s['boxes'].copy(); moved[:, [0, 2]] += 40          # 大幅平移 -> 大量越界
kept, dropped = clip_and_filter(moved, W, H)
print('平移后 %d 个框 -> 清洗后保留 %d 个，丢弃 %d 个' % (len(moved), len(kept), dropped))
assert dropped > 0 and len(kept) + dropped == len(moved)
if len(kept):
    w = kept[:, 2] - kept[:, 0]
    print('保留框的最小宽度: %.1f px（阈值挡住了 <4px 的贴边窄条）' % w.min())
    assert (kept[:, 2] - kept[:, 0]).min() * (kept[:, 3] - kept[:, 1]).min() >= 0
print()
print('⚠️  若把清洗放到流水线末尾、中间又插了 Mosaic 的二次拼接，')
print('   裁剪就作用在**错误的画布**上 -> 训练集里凭空出现大量贴边窄条框，')
print('   模型学到"标志可以只有 2 像素宽"，直接污染小目标的尺度先验。')
print('✅ 规则：坐标系一变，立刻清洗。')

## 4 · DataLoader worker 的 RNG 陷阱（真实事故）

DataLoader 的 worker 是 **fork** 出来的：numpy / python-random 的全局状态被**原样复制**。
PyTorch 只替 torch 自己的 RNG 做了 `base_seed + worker_id`，**numpy 不在此列**——
而绝大多数检测增强库用的正是 numpy。

下面用三种 seed 策略跑同一个 epoch，看增强参数的**去重个数**。

In [ ]:
NUM_WORKERS, N_SAMPLES = 4, 64

def epoch_params(strategy, base_seed, num_workers, n, epoch):
    '''模拟一个 epoch：样本 i 由 worker (i % W) 处理，按顺序从该 worker 的 RNG 取增强参数。'''
    params = [None] * n
    for w in range(num_workers):
        if strategy == 'bad':            # ❌ fork 后全 worker 共享同一状态
            g = np.random.default_rng(base_seed)
        elif strategy == 'worker_only':  # ⚠️ 加了 worker_id 但没加 epoch
            g = np.random.default_rng(base_seed + w)
        elif strategy == 'good':         # ✅ base_seed + epoch + worker_id
            g = np.random.default_rng(
                np.random.SeedSequence(entropy=base_seed, spawn_key=(epoch, w)))
        else:
            raise ValueError(strategy)
        for i in range(w, n, num_workers):
            params[i] = round(float(g.random()), 12)     # 一个样本的增强参数
    return params

p_bad = epoch_params('bad', 1234, NUM_WORKERS, N_SAMPLES, 0)
p_wo = epoch_params('worker_only', 1234, NUM_WORKERS, N_SAMPLES, 0)
p_good = epoch_params('good', 1234, NUM_WORKERS, N_SAMPLES, 0)

print('一个 epoch，%d 个样本，%d 个 worker' % (N_SAMPLES, NUM_WORKERS))
print('%-16s %-16s %s' % ('策略', '不同参数个数', '结论'))
print('%-16s %-16d %s' % ('❌ bad', len(set(p_bad)), '多样性 ÷ %d' % NUM_WORKERS))
print('%-16s %-16d %s' % ('⚠️ worker_only', len(set(p_wo)), 'epoch 内 OK'))
print('%-16s %-16d %s' % ('✅ good', len(set(p_good)), 'epoch 内 OK'))
assert len(set(p_bad)) == N_SAMPLES // NUM_WORKERS == 16
assert len(set(p_wo)) == N_SAMPLES and len(set(p_good)) == N_SAMPLES
print()
print('样本 0/1/2/3 分别由 worker 0/1/2/3 处理，bad 策略下它们的参数:')
print('  ', [round(x, 6) for x in p_bad[:4]], '<- 完全相同')
print('   good 策略下:')
print('  ', [round(x, 6) for x in p_good[:4]], '<- 各不相同')
assert len(set(p_bad[:NUM_WORKERS])) == 1
assert len(set(p_good[:NUM_WORKERS])) == NUM_WORKERS

In [ ]:
# 跨 epoch 的后果：seed 里没有 epoch，整个训练就在重复同一批增强
EPOCHS = 50
totals = {}
for strat in ('bad', 'worker_only', 'good'):
    seen = set()
    for ep in range(EPOCHS):
        seen |= set(epoch_params(strat, 1234, NUM_WORKERS, N_SAMPLES, ep))
    totals[strat] = len(seen)

print('训练 %d 个 epoch，模型总共见过多少种不同的增强参数？' % EPOCHS)
print('%-16s %14s %14s' % ('策略', '总去重个数', '相对正确实现'))
for k, v in totals.items():
    print('%-16s %14d %13.2f%%' % (k, v, 100 * v / totals['good']))
assert totals['bad'] == 16, 'bad: seed 与 worker/epoch 都无关 -> 全程只有 16 种'
assert totals['worker_only'] == N_SAMPLES, 'worker_only: 每个 epoch 完全重复'
assert totals['good'] == N_SAMPLES * EPOCHS
print()
print('❌ bad        : 全程 16 种增强 —— 多样性是正确实现的 %.2f%%' % (100 * 16 / totals['good']))
print('⚠️ worker_only: 每个 epoch 一模一样，训练 50 轮等于只有 1 轮的多样性')
print('✅ good       : 64 × 50 = 3200 种')
print()
print('关键：以上三种都**不会报错、不会 warning、loss 曲线完全正常**。')
print('     你看到的只是"mAP 比论文低两个点"，然后去怪超参和 backbone。')

# 可复现性：同 seed/epoch 必须逐位一致
assert epoch_params('good', 1234, 4, 64, 3) == epoch_params('good', 1234, 4, 64, 3)
# 但换 num_workers 会改变序列 —— 所以复现记录里必须写 num_workers
assert epoch_params('good', 1234, 4, 64, 3) != epoch_params('good', 1234, 8, 64, 3)
print()
print('✅ 同 (seed, epoch, num_workers) -> 逐位可复现')
print('⚠️ 换 num_workers -> 序列改变。"可复现"必须记录 seed + num_workers + 配置哈希 + 库版本。')

In [ ]:
# 把防线写成一条断言：一个 epoch 内增强参数的去重个数必须等于样本数
def assert_aug_diversity(params, n_samples, name=''):
    u = len(set(params))
    if u != n_samples:
        raise AssertionError(
            'worker RNG 疑似共享: 去重参数 %d != 样本数 %d（多样性打了 1/%.1f 折）'
            % (u, n_samples, n_samples / max(u, 1)))
    return True

assert assert_aug_diversity(p_good, N_SAMPLES)
try:
    assert_aug_diversity(p_bad, N_SAMPLES)
    raise RuntimeError('不该走到这里')
except AssertionError as e:
    print('✅ 断言成功拦截:', e)
print()
print('这十行代码写一次、终身受益 —— 把一类"永远查不出原因"的事故变成启动即报错。')

## 5 · 吞吐瓶颈：CPU 供给 vs GPU 需求

`throughput = min(W / t_aug, B / t_step)`。
先算出瓶颈在哪一侧，再决定拧哪个旋钮——**不要先动手优化，先做这道算术。**

In [ ]:
def throughput(t_aug_ms, num_workers, gpu_step_ms, batch):
    cpu_ips = num_workers / (t_aug_ms / 1000.0)      # CPU 侧供给 (img/s)
    gpu_ips = batch / (gpu_step_ms / 1000.0)         # GPU 侧需求 (img/s)
    return {'cpu_ips': cpu_ips, 'gpu_ips': gpu_ips,
            'ips': min(cpu_ips, gpu_ips),
            'gpu_util': min(1.0, cpu_ips / gpu_ips),
            'bottleneck': 'CPU(数据增强)' if cpu_ips < gpu_ips else 'GPU(前反向)'}

def min_workers(t_aug_ms, gpu_step_ms, batch):
    '''喂饱 GPU 所需的最少 worker 数。'''
    return int(math.ceil((batch / (gpu_step_ms / 1000.0)) * (t_aug_ms / 1000.0)))

T_STEP, BATCH = 60.0, 32
print('GPU 需求: batch=%d / step=%.0fms = %.1f img/s' % (BATCH, T_STEP, BATCH / (T_STEP / 1000)))
print()
print('%-30s %11s %11s %-16s %9s' % ('配置', 'CPU供给', 'GPU需求', '瓶颈', 'GPU利用率'))
for t_aug, W, tag in [(12.0, 4, '基础增强, W=4'), (12.0, 7, '基础增强, W=7'),
                      (12.0, 8, '基础增强, W=8'), (38.4, 8, '开 Mosaic(≈3.2x), W=8'),
                      (38.4, 21, '开 Mosaic, W=21')]:
    r = throughput(t_aug, W, T_STEP, BATCH)
    print('%-30s %11.0f %11.0f %-16s %8.1f%%'
          % (tag, r['cpu_ips'], r['gpu_ips'], r['bottleneck'], 100 * r['gpu_util']))

assert throughput(12, 4, T_STEP, BATCH)['bottleneck'].startswith('CPU')
assert throughput(12, 8, T_STEP, BATCH)['gpu_util'] == 1.0
assert abs(throughput(12, 4, T_STEP, BATCH)['gpu_util'] - 0.625) < 1e-9
assert min_workers(12.0, T_STEP, BATCH) == 7
assert min_workers(38.4, T_STEP, BATCH) == 21
assert min_workers(5.0, T_STEP, BATCH) == 3
print()
print('喂饱一张卡所需最少 worker: 基础增强 %d 个；开 Mosaic 后 %d 个。'
      % (min_workers(12.0, T_STEP, BATCH), min_workers(38.4, T_STEP, BATCH)))
print('⚠️  一台 8 卡机开 Mosaic 就需要 %d 核 —— 绝大多数机器给不出来。'
      % (8 * min_workers(38.4, T_STEP, BATCH)))
print('   这就是"开了 Mosaic 之后训练明显变慢却查不出原因"的算术解释。')

In [ ]:
# 预取只削峰，不提均值：用 max-plus 递推精确验证
def simulate(prep_times, t_gpu, prefetch):
    '''prefetch 个 batch 在 t=0 已就绪；其余按生产顺序陆续到达。'''
    cum = np.cumsum(prep_times)
    n = len(prep_times)
    end, idle_total = 0.0, 0.0
    for i in range(n):
        avail = 0.0 if i < prefetch else float(cum[i - prefetch])
        idle_total += max(0.0, avail - end)
        end = max(end, avail) + t_gpu
    return {'wall': end, 'idle': idle_total, 'per_step': end / n}

def lognormal_times(rng, n, mean, sigma=0.25):
    mu = math.log(mean) - sigma ** 2 / 2
    return np.exp(rng.normal(mu, sigma, size=n))

rg = np.random.default_rng(11)
N_B, T_GPU = 1000, 0.060

prep_A = lognormal_times(rg, N_B, 0.048)      # 场景 A：CPU 比 GPU 快 25%
prep_B = lognormal_times(rg, N_B, 0.075)      # 场景 B：CPU 比 GPU 慢 25%

print('%-38s %10s %10s %10s' % ('场景 / 预取深度', '每步(ms)', '空转(s)', '相对GPU下限'))
rows = []
for tag, prep in [('A: CPU快25% (mean 48ms)', prep_A), ('B: CPU慢25% (mean 75ms)', prep_B)]:
    for q in (0, 8, 64):
        r = simulate(prep, T_GPU, q)
        rows.append((tag, q, r))
        print('%-38s %10.2f %10.2f %9.2fx'
              % ('%s | q=%d' % (tag, q), 1000 * r['per_step'], r['idle'],
                 r['per_step'] / T_GPU))

A0, A8, A64 = [r for t, q, r in rows if t.startswith('A')]
B0, B8, B64 = [r for t, q, r in rows if t.startswith('B')]
# 单调性：预取只会让空转不增（max-plus 递推的单调性，恒成立）
assert A0['idle'] >= A8['idle'] >= A64['idle']
assert B0['idle'] >= B8['idle'] >= B64['idle']
# 场景 A：CPU 够快，q=8 就已经把 GPU 喂满，每步 = t_gpu
assert abs(A8['per_step'] - T_GPU) < 1e-3, 'CPU 够快时预取能把利用率拉到 100%'
# 场景 B：CPU 不够快，加深预取也救不了
assert B8['per_step'] > T_GPU * 1.10 and B64['per_step'] > T_GPU * 1.10
print()
print('场景 A：q 从 0 -> 8，空转 %.2fs -> %.2fs，每步稳定在 %.1fms = GPU 下限。'
      % (A0['idle'], A8['idle'], 1000 * A8['per_step']))
print('场景 B：q 从 8 -> 64（8 倍队列），每步只从 %.2fms 降到 %.2fms（%.1f%%）。'
      % (1000 * B8['per_step'], 1000 * B64['per_step'],
         100 * (B8['per_step'] - B64['per_step']) / B8['per_step']))
print()
print('✅ 结论：prefetch 是**方差工具**不是**均值工具**。')
print('   把 prefetch_factor 从 2 调到 16 却没变快 -> 你面对的是均值问题，')
print('   唯一出路是降低 t_aug（预解码/合并算子/关库内多线程）或提高 worker 数。')

## 6 · TTA 与 WBF：多个视角怎么融合

TTA 的 6 个分支给出的是**同一目标的 6 次独立观测**。
NMS 的语义是"选一个删其他"——等于扔掉 5 份可以用来降噪的信息。
WBF 的语义是"分数加权平均 + 按支持分支数重标定分数"。

In [ ]:
def iou1(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def nms(boxes, scores, thr=0.55):
    '''返回保留框的下标（按分数降序）。'''
    order = [int(i) for i in np.argsort(-np.asarray(scores))]
    keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou1(boxes[i], boxes[j]) < thr]
    return keep

def wbf(boxes, scores, n_branches, iou_thr=0.55):
    '''加权框融合：坐标按分数加权平均，分数 = 簇内均分 × min(n, T)/T。'''
    boxes = np.asarray(boxes, float); scores = np.asarray(scores, float)
    clusters = []
    for idx in np.argsort(-scores):
        b, s = boxes[idx].copy(), float(scores[idx])
        best, best_iou = -1, iou_thr
        for ci, c in enumerate(clusters):
            v = iou1(b, c['fused'])
            if v > best_iou:
                best, best_iou = ci, v
        if best < 0:
            clusters.append({'b': [b], 's': [s], 'fused': b.copy()})
        else:
            c = clusters[best]; c['b'].append(b); c['s'].append(s)
            w = np.asarray(c['s']); B = np.asarray(c['b'])
            c['fused'] = (B * w[:, None]).sum(0) / w.sum()     # ← 分数加权平均坐标
    fb = np.asarray([c['fused'] for c in clusters])
    cnt = np.asarray([len(c['s']) for c in clusters])
    fs = np.asarray([float(np.mean(c['s'])) * min(len(c['s']), n_branches) / n_branches
                     for c in clusters])                        # ← 少数分支支持 -> 降分
    return fb, fs, cnt

# 手算校验
bx = np.array([[0., 0., 10., 10.], [1., 1., 11., 11.], [100., 100., 110., 110.]])
sc = np.array([0.9, 0.7, 0.8])
print('IoU(box0, box1) = %.4f' % iou1(bx[0], bx[1]))
fb, fs, cnt = wbf(bx, sc, n_branches=3, iou_thr=0.5)
print('簇大小:', cnt.tolist())
print('融合框 0:', np.round(fb[0], 4).tolist(), ' 分数 %.6f' % fs[0])
print('融合框 1:', np.round(fb[1], 4).tolist(), ' 分数 %.6f' % fs[1])
assert cnt.tolist() == [2, 1]
assert np.allclose(fb[0], [0.4375, 0.4375, 10.4375, 10.4375])   # (0.9*b0+0.7*b1)/1.6
assert abs(fs[0] - 0.8 * 2 / 3) < 1e-12 and abs(fs[1] - 0.8 * 1 / 3) < 1e-12
print('✅ 只被 1/3 分支支持的框，分数被打到 1/3 —— NMS 对这类偶发误检无能为力。')

In [ ]:
# 定位精度对比：3 个 TTA 分支各带独立定位噪声
GT = np.array([100., 100., 150., 150.])
T_BR, TRIALS = 3, 2000
rq = np.random.default_rng(3)
iou_nms, iou_wbf = [], []
for _ in range(TRIALS):
    bs = GT + rq.normal(0, 6.0, size=(T_BR, 4))       # 每个分支独立的定位噪声
    ss = rq.uniform(0.55, 0.95, size=T_BR)
    keep = nms(bs, ss, thr=0.55)
    iou_nms.append(iou1(bs[keep[0]], GT))             # NMS: 就是"最高分那一个"
    fb, fs, _ = wbf(bs, ss, n_branches=T_BR, iou_thr=0.55)
    iou_wbf.append(iou1(fb[int(np.argmax(fs))], GT))

m_nms, m_wbf = float(np.mean(iou_nms)), float(np.mean(iou_wbf))
print('%d 次试验，3 个 TTA 分支，每坐标 sigma=6px（目标 50x50）' % TRIALS)
print('  NMS 融合（=选最高分）  平均 IoU = %.4f' % m_nms)
print('  WBF 融合（=加权平均）  平均 IoU = %.4f' % m_wbf)
print('  提升 = %+.4f' % (m_wbf - m_nms))
assert m_wbf > m_nms + 0.02, 'n 次独立观测取平均，定位方差降到 1/n'
print()
print('原理：n 次独立观测的加权平均，其方差是单次的 1/n（这里 3 个分支 -> sigma 6 -> 3.5）。')
print('NMS 删掉其余 %d 个观测，定位方差一点没降；WBF 把它们全用上了。' % (T_BR - 1))
print('✅ 所以融合多个 TTA / 多模型结果时用 WBF，不要用 NMS。')

In [ ]:
# 车端延迟账：TTA 为什么上不了车
BASE_MS = 8.0                       # 单次推理
SCALES = [0.83, 1.0, 1.25]
FLIPS = 2
FUSION_MS = 1.5

per_scale = [BASE_MS * s * s for s in SCALES]        # 计算量 ∝ 面积
tta_ms = sum(per_scale) * FLIPS + FUSION_MS

print('单次推理           : %.1f ms' % BASE_MS)
for s, c in zip(SCALES, per_scale):
    print('  尺度 %.2f 分支    : %.2f ms' % (s, c))
print('3 尺度 × 2 翻转     : %.2f ms' % (sum(per_scale) * FLIPS))
print('WBF 融合            : %.1f ms' % FUSION_MS)
print('TTA 总计            : %.2f ms  = %.2fx 单次推理' % (tta_ms, tta_ms / BASE_MS))
print()
FRAME_MS = 1000.0 / 30
TSR_BUDGET_MS = 10.0
print('车端预算: 30 FPS -> 每帧 %.1f ms，要装下多路感知+融合+跟踪+规控' % FRAME_MS)
print('          TSR 分到的通常只有 %.0f ms' % TSR_BUDGET_MS)
print('TTA 超出 TSR 预算 %.1f 倍；超出整帧预算 %.1f 倍'
      % (tta_ms / TSR_BUDGET_MS, tta_ms / FRAME_MS))
assert tta_ms > 5 * TSR_BUDGET_MS
assert tta_ms > FRAME_MS
print()
print('⚠️  更致命的是 TTA 的延迟**随目标数波动**（融合是 O(n^2)），破坏 p99 的确定性 ——')
print('    而车端要的恰恰是确定性。**结论：TTA 在量产感知里基本不可用。**')
print('✅ TTA 该出现的地方：① 打伪标签  ② 难例挖掘的一致性信号  ③ 估计模型能力上限')
print('   （TTA 涨得越多，说明你的增强训练做得越差 —— 这是一个诊断信号）')

## ✏️ 练习 1：联合触发分布与概率反解

实现两个函数：
- `trigger_profile_exact(ps)`：精确返回"恰好触发 j 个"的概率数组（长度 `len(ps)+1`），**不许用模拟**。
- `calibrate_uniform_p(k, target_all)`：k 个同概率算子，希望"全部触发"的样本占 `target_all`，返回每个算子的 p。

In [ ]:
def trigger_profile_exact(ps):
    # TODO: 枚举 2^k 种触发组合，把概率累加到"触发个数"这一维
    raise NotImplementedError

def calibrate_uniform_p_ex(k, target_all):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
d = trigger_profile_exact([0.5] * 5)
assert len(d) == 6 and abs(sum(d) - 1.0) < 1e-12
assert abs(d[5] - 1 / 32) < 1e-12 and abs(d[0] - 1 / 32) < 1e-12
assert abs(float(np.dot(np.arange(6), d)) - 2.5) < 1e-12
d2 = trigger_profile_exact([0.9, 0.2, 0.5])
assert abs(d2[3] - 0.9 * 0.2 * 0.5) < 1e-12
assert abs(d2[0] - 0.1 * 0.8 * 0.5) < 1e-12
d3 = trigger_profile_exact([1.0, 0.5])
assert abs(d3[0] - 0.0) < 1e-12 and abs(d3[2] - 0.5) < 1e-12
p = calibrate_uniform_p_ex(5, 0.30)
assert abs(p ** 5 - 0.30) < 1e-12 and abs(p - 0.30 ** 0.2) < 1e-12
assert abs(calibrate_uniform_p_ex(8, 0.5) ** 8 - 0.5) < 1e-12
print('8 个算子想让 50%% 的样本走完全套 -> 每个 p = %.4f'
      % calibrate_uniform_p_ex(8, 0.5))
print('✅ 练习 1 通过：**先定联合强度，再反解每个 p**，而不是逐个算子拍脑袋')

## ✏️ 练习 2：正确的 worker seed 策略

实现 `epoch_draws(base_seed, epoch, num_workers, n)`：模拟一个 epoch，
样本 `i` 由 worker `i % num_workers` 处理，返回长度 n 的增强参数列表。
要求：**同一个 epoch 内 n 个参数互不相同；不同 epoch 之间也互不相同；
同 (seed, epoch, num_workers) 逐位可复现。**

In [ ]:
def epoch_draws(base_seed, epoch, num_workers, n):
    # TODO: 每个 worker 用 np.random.SeedSequence(entropy=..., spawn_key=(epoch, worker_id))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
a = epoch_draws(1234, 0, 4, 64)
b = epoch_draws(1234, 1, 4, 64)
c = epoch_draws(1234, 0, 4, 64)
assert len(a) == 64 and len(set(a)) == 64, 'epoch 内必须 64 个各不相同'
assert len(set(a) & set(b)) == 0, '不同 epoch 不能重复'
assert a == c, '同 (seed, epoch, num_workers) 必须逐位可复现'
d8 = epoch_draws(1234, 0, 8, 64)
assert len(set(d8)) == 64
assert d8 != a, '换 num_workers 会改变序列 —— 复现记录必须写下 num_workers'
assert_aug_diversity(a, 64)
print('epoch0 前 4 个参数:', [round(x, 6) for x in a[:4]])
print('epoch1 前 4 个参数:', [round(x, 6) for x in b[:4]])
print('✅ 练习 2 通过：seed 必须同时含 **base_seed / epoch / worker_id** 三者')

## ✏️ 练习 3：TTA 分支的逆变换

每个 TTA 分支在"缩放 + 可选翻转"后的图上推理，框必须还原到原图坐标系。
实现 `tta_untransform(boxes, scale, flipped, W)`：`boxes` 是分支输出的 `(N,4) xyxy`，
`scale` 是该分支相对原图的缩放比，`flipped` 表示该分支做过水平翻转，`W` 是**原图宽度**。

⚠️ 翻转的逆变换必须 **交换 x1/x2**，否则得到 `x2 < x1` 的非法框（IoU 恒为 0，症状是"TTA 之后反而掉点"）。

In [ ]:
def tta_untransform(boxes, scale, flipped, W):
    # TODO: ① 先除以 scale 回到原图尺寸  ② 若 flipped，用 W 做镜像并**交换 x1/x2**
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
b0 = np.array([[10., 20., 30., 40.]])
o1 = tta_untransform(b0, scale=2.0, flipped=False, W=200)
assert np.allclose(o1, [[5., 10., 15., 20.]]), o1
o2 = tta_untransform(b0, scale=1.0, flipped=True, W=100)
assert np.allclose(o2, [[70., 20., 90., 40.]]), o2      # x1=100-30, x2=100-10
o3 = tta_untransform(b0, scale=2.0, flipped=True, W=100)
assert np.allclose(o3, [[85., 10., 95., 20.]]), o3      # 先 /2 -> [5,10,15,20]，再镜像
for o in (o1, o2, o3):
    assert (o[:, 2] > o[:, 0]).all() and (o[:, 3] > o[:, 1]).all(), '必须是合法框'
# 忘记交换 x1/x2 的后果
wrong = b0.copy(); wrong[:, 0] = 100 - wrong[:, 0]; wrong[:, 2] = 100 - wrong[:, 2]
assert (wrong[:, 2] < wrong[:, 0]).all()
assert iou1(wrong[0], np.array([70., 20., 90., 40.])) == 0.0
print('忘记交换 x1/x2 得到:', wrong.tolist(), ' -> 与正确框的 IoU =',
      iou1(wrong[0], o2[0]))
print('✅ 练习 3 通过：翻转分支忘记交换 x1/x2 -> 非法框 -> IoU 恒 0 -> "TTA 反而掉点"')

## ✏️ 练习 4：worker 数规划

实现 `plan_workers(t_aug_ms, gpu_step_ms, batch, cpu_cores)`，返回
`{'cpu_ips','gpu_ips','min_workers','feasible','bottleneck'}`。
`min_workers` = 喂饱 GPU 所需最少 worker 数（向上取整）；
`feasible` = `min_workers <= cpu_cores`；`bottleneck` 按 `cpu_cores` 全开时判断。

In [ ]:
def plan_workers(t_aug_ms, gpu_step_ms, batch, cpu_cores):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
r1 = plan_workers(12.0, 60.0, 32, 16)
assert r1['min_workers'] == 7 and r1['feasible']
assert abs(r1['gpu_ips'] - 533.3333333) < 1e-4
r2 = plan_workers(38.4, 60.0, 32, 16)
assert r2['min_workers'] == 21 and not r2['feasible']
assert r2['bottleneck'].startswith('CPU')
r3 = plan_workers(5.0, 60.0, 32, 16)
assert r3['min_workers'] == 3 and r3['feasible'] and r3['bottleneck'].startswith('GPU')
print('%-34s %8s %8s %10s' % ('配置', '最少W', '可行?', '瓶颈'))
for tag, t, cores in [('基础增强, 16 核', 12.0, 16), ('开 Mosaic, 16 核', 38.4, 16),
                      ('开 Mosaic, 32 核', 38.4, 32), ('预解码后, 16 核', 5.0, 16)]:
    r = plan_workers(t, 60.0, 32, cores)
    print('%-34s %8d %8s %10s' % (tag, r['min_workers'],
                                  '✅' if r['feasible'] else '❌', r['bottleneck']))
print('✅ 练习 4 通过：开增强之前先做这道算术，别等训练慢了再查')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def trigger_profile_exact(ps):
    k = len(ps)
    dist = np.zeros(k + 1)
    for mask in itertools.product([0, 1], repeat=k):
        pr = 1.0
        for bit, p in zip(mask, ps):
            pr *= p if bit else (1.0 - p)
        dist[sum(mask)] += pr
    return dist

def calibrate_uniform_p_ex(k, target_all):
    return target_all ** (1.0 / k)

In [ ]:
# 练习 2 参考答案
def epoch_draws(base_seed, epoch, num_workers, n):
    out = [None] * n
    for w in range(num_workers):
        ss = np.random.SeedSequence(entropy=base_seed, spawn_key=(epoch, w))
        g = np.random.default_rng(ss)
        for i in range(w, n, num_workers):
            out[i] = round(float(g.random()), 12)
    return out

In [ ]:
# 练习 3 参考答案
def tta_untransform(boxes, scale, flipped, W):
    b = np.asarray(boxes, float) / float(scale)      # ① 回到原图尺寸
    if flipped:                                       # ② 镜像并交换 x1/x2
        x1, x2 = b[:, 0].copy(), b[:, 2].copy()
        b[:, 0], b[:, 2] = W - x2, W - x1
    return b

In [ ]:
# 练习 4 参考答案
def plan_workers(t_aug_ms, gpu_step_ms, batch, cpu_cores):
    gpu_ips = batch / (gpu_step_ms / 1000.0)
    need = int(math.ceil(gpu_ips * (t_aug_ms / 1000.0)))
    cpu_ips = cpu_cores / (t_aug_ms / 1000.0)
    return {'cpu_ips': cpu_ips, 'gpu_ips': gpu_ips, 'min_workers': need,
            'feasible': need <= cpu_cores,
            'bottleneck': 'CPU(数据增强)' if cpu_ips < gpu_ips else 'GPU(前反向)'}

---
## 🧪 真实工程胶囊：一份可直接搬走的流水线配置与自检

In [ ]:
RECIPE = r'''
# ============ 1. DataLoader：seed / 线程 / 预取 ============
import cv2, numpy as np, random, torch
cv2.setNumThreads(0)                    # ★ 第一条：关掉库内多线程，避免 W×N 线程风暴

def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 31)   # = base_seed + worker_id，且**每 epoch 变**
    np.random.seed(seed); random.seed(seed)   # ★ torch 会自己处理，numpy/random 不会
    cv2.setNumThreads(0)

loader = torch.utils.data.DataLoader(
    ds, batch_size=32, shuffle=True,
    num_workers=8,                       # ★ 先算 min_workers = ceil(gpu_ips * t_aug)
    worker_init_fn=worker_init_fn,
    generator=torch.Generator().manual_seed(1234),
    persistent_workers=True,             # 避免每 epoch 重启 worker
    prefetch_factor=4,                   # 只削峰，不提均值
    pin_memory=True, drop_last=True,
)

# ============ 2. 增强顺序：混合 -> 几何 -> 清洗 -> 光度 -> 归一 ============
import albumentations as A
GEOM = A.Compose([                                   # ★ 几何段合成一次仿射，只重采样一次
    A.Affine(scale=(0.5, 1.5), translate_percent=(-0.1, 0.1),
             rotate=(-8, 8), shear=(-4, 4), p=0.9),
    A.HorizontalFlip(p=0.5),                         # ★ TSR: 必须配类别白名单（左转/右转！）
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'],
                            min_area=16, min_visibility=0.3))   # ★ 清洗紧跟几何
PHOTO = A.Compose([                                  # ★ 光度段放在几何之后，否则被插值抹平
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=25,
                         val_shift_limit=25, p=0.7), # ★ TSR: hue 幅度必须 <=10（颜色是语义）
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.OneOf([A.MotionBlur(7), A.GaussNoise((5, 30)), A.ImageCompression(40, 85)], p=0.4),
])
# 验证/部署路径：只保留确定性段，**与训练共用同一个 letterbox 函数**
VAL = A.Compose([])   # 仅 letterbox + normalize，写在 collate 之外，train/val/deploy 共享

# ============ 3. 启动自检：三条断言，写一次终身受益 ============
def startup_checks(dataset, loader, n_probe=2000):
    # (a) 实际触发率：不要读配置推理，要打点
    hits = {}
    for i in range(n_probe):
        for name in dataset.last_fired_ops(i):    # 你的 Pipeline 需要记录 fired
            hits[name] = hits.get(name, 0) + 1
    rates = {k: v / n_probe for k, v in hits.items()}
    print('实测触发率:', {k: round(v, 4) for k, v in sorted(rates.items())})
    joint = np.prod([v for v in rates.values()])
    print('联合(全触发)概率: %.4f  <- 常常比你以为的低一个数量级' % joint)

    # (b) worker RNG 多样性：一个 epoch 的增强参数去重数必须等于样本数
    params = collect_aug_params_one_epoch(loader)
    assert len(set(params)) == len(params),         'worker RNG 疑似共享！多样性打了 1/num_workers 的折扣'

    # (c) 评测路径必须是纯确定性的：同权重评两次必须逐位相同
    assert evaluate(model, val_loader) == evaluate(model, val_loader),         '评测路径残留随机性 -> 所有消融/显著性结论作废'

# ============ 4. 配置指纹：写进 checkpoint 与每行日志 ============
import hashlib, json
def aug_fingerprint(cfg, lib_versions):
    blob = json.dumps({'cfg': cfg, 'lib': lib_versions}, sort_keys=True)
    return hashlib.sha1(blob.encode()).hexdigest()[:12]
# 记录: seed / num_workers / 算子顺序+参数+p / 实测触发率 /
#       close_mosaic_epoch / cv2,PIL,albumentations 版本 / 指纹

# ============ 5. TTA（仅离线：伪标签 / 难例挖掘 / 能力上限）============
# 分支: scales [0.83,1.0,1.25] × flip [F,T] = 6 次前向 ≈ 6.5x 延迟
# 每个分支输出后先 tta_untransform（★ 翻转要交换 x1/x2），再 WBF 融合：
#   b_fused = sum(s_i*b_i)/sum(s_i);  s_fused = mean(s)*min(n,T)/T
# 车端预算 8-12ms -> TTA 53ms，超 5 倍且延迟随目标数波动 -> **不上车**
'''
print(RECIPE)
for k in ['setNumThreads(0)', 'worker_init_fn', 'torch.initial_seed', 'persistent_workers',
          'min_visibility', 'hue_shift_limit', 'aug_fingerprint', 'tta_untransform',
          'min(n,T)/T', 'num_workers']:
    assert k in RECIPE, k
print('✅ 配方覆盖：线程/seed/预取 · 顺序与清洗 · 三条启动断言 · 配置指纹 · TTA 与 WBF')

### 小结

- **顺序**：混合 → 几何（合成一个矩阵、只重采样一次）→ 立刻清洗越界框 → 光度 → 与部署逐位一致的归一化。
  光度放在几何之前，注入的噪声会被插值抹掉 **50%–69%**；几何串联 3 次比合成 1 次多丢 **37.5%** 的高频能量。
- **概率相乘**：5 个算子各 p=0.5，全部触发只有 **3.1%**；8 个各 0.5 只有 **0.39%**。
  想要 30% 的样本走完 5 个算子，每个 p 要 **0.786**。嵌套 `OneOf` 会把叶子触发率再压一个数量级。
  **不要读配置推理，要打点测量。**
- **worker RNG 陷阱**：fork 复制 numpy 全局状态 → 所有 worker 生成**完全相同**的增强序列，
  多样性 ÷ num_workers，**且不报错**。seed 必须同时含 `base_seed / epoch / worker_id`。
  防线是一条断言：**一个 epoch 内增强参数的去重数必须等于样本数。**
- **吞吐**：`min(W/t_aug, B/t_step)`。开 Mosaic 后喂饱一张卡要 21 个 worker。
  **prefetch 只削峰不提均值**——调深了没变快，说明是均值问题，得降 `t_aug`。
- **train/val/deploy 的确定性段必须是同一份代码**；评测路径必须纯确定性，否则消融结论建立在流沙上。
- **TTA 融合用 WBF 不用 NMS**：n 次独立观测取加权平均，定位方差降到 1/n；
  再按支持分支数重标定分数，天然抑制单分支误检。**但 6.5× 延迟 + p99 不确定 → 车端不可用**，
  它的正确用途是打伪标签、难例挖掘的一致性信号、以及估计模型能力上限。

下一站：**模块 05 · 增强的消融与验证** —— 有了这条流水线，怎么证明它真的有用。